# Comprehensive Skill Gap Analysis Pipeline
This notebook analyzes Job Descriptions (JDs) and Resumes to identify the skill gap for candidates. 

It combines two powerful extraction engines:
1. **JD Extraction (TrueHybridMatcher)**: Uses a dual-engine (Dictionary + Custom ML NER) to extract required skills, dynamically assess their difficulty, and calculate a continuous Priority Score geared towards creating a step-by-step learning roadmap (easier/foundational skills get higher priority).
2. **Resume Extraction (SkillNER)**: Uses the `skillNer` library combined with `spaCy` to robustly extract a candidate's existing skill set from their resume.

Finally, it calculates the set difference (Missing Skills) and outputs a structured JSON report.

In [1]:
# Environment Setup and Library Imports
import pandas as pd
import spacy
import re
import json
import warnings
from typing import List, Dict, Set
from tqdm import tqdm
import pickle
import os
from tqdm import tqdm

# 1. Mute SpaCy warnings regarding empty vectors to keep the output clean
warnings.filterwarnings("ignore", message=".*empty vectors.*")
warnings.filterwarnings("ignore", category=UserWarning)

# 2. Enable pandas progress bar for large datasets
tqdm.pandas()

# 3. SkillNER imports
from spacy.matcher import PhraseMatcher
from skillNer.general_params import SKILL_DB
from skillNer.skill_extractor_class import SkillExtractor

print("✅ Phase 1: Environment configured successfully. Warnings suppressed.")

✅ Phase 1: Environment configured successfully. Warnings suppressed.


## Phase 1: Environment Setup & Model Loading
We will load the base NLP models, the Custom NER model (for JDs), and the SkillNER extractor (for Resumes).

In [2]:
print("Loading NLP models and Custom Vocabulary...")

# OPTIMIZATION 1: Disable the 'ner' (Named Entity Recognition) pipeline for the base model.
# Since we only need sentence segmentation and basic tokenization, disabling 'ner'
# drastically speeds up the processing time.
nlp_base = spacy.load("en_core_web_sm", disable=["ner"])

# Large Model for SkillNER (Resumes)
try:
    # SkillNER internally uses PhraseMatcher. We can safely disable 'ner' and 'parser' 
    # to save memory and parsing time for the resume extraction phase.
    nlp_skillner = spacy.load("en_core_web_lg", disable=["ner", "parser"])
except OSError:
    print("⚠️ Using small model for SkillNER (run !python -m spacy download en_core_web_lg to upgrade).")
    nlp_skillner = spacy.load("en_core_web_sm", disable=["ner", "parser"])
    
skill_extractor = SkillExtractor(nlp_skillner, SKILL_DB, PhraseMatcher)

# Load the Semi-Supervised Expanded Vocabulary (Engine 2)
# Ensure this path is absolutely correct for your local machine
custom_model_path = "/Users/fengmengting/Desktop/berkeley/26spring/INDENG243 Analytics Lab/project/module2_new/job_post_extraction/hard_skill_model.pkl"
try:
    with open(custom_model_path, "rb") as f:
        custom_data = pickle.load(f)
        # Extract the set of 5000+ skills discovered by Word2Vec
        expanded_vocab = custom_data.get('vocab', set()) 
    print(f"✅ Custom Word2Vec Model loaded! Expanded vocab size: {len(expanded_vocab)} skills.")
except Exception as e:
    print(f"⚠️ Custom model failed to load. ERROR: {e}")
    expanded_vocab = set()

print("✅ Phase 2 Complete. All Models and Vocabularies are ready.")

Loading NLP models and Custom Vocabulary...
loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...
✅ Custom Word2Vec Model loaded! Expanded vocab size: 5296 skills.
✅ Phase 2 Complete. All Models and Vocabularies are ready.


## Phase 2: Define the JD Skill Extractor Engine
This class uses a True Dual-Engine approach:
* **Engine 1**: The canonical `SKILL_TAXONOMY` (with alias resolution).
* **Engine 2**: The Word2Vec Expanded Vocabulary (5200+ skills from `ner_train`).

This class extracts required skills from the Job Description. It incorporates our optimized formula for learning roadmaps:
`Priority Score = Relevance_Multiplier * (5 - Difficulty)`
where lower difficulty (Basic) yields a higher score, encouraging foundational learning first.

In [3]:
SKILL_TAXONOMY: dict[str, dict] = {
    # ── Programming Languages ──────────────────
    "python":           {"category": "Programming",      "difficulty": 2, "aliases": ["python3", "python2", "py"]},
    "r":                {"category": "Programming",      "difficulty": 2, "aliases": ["r programming", "r-studio", "rstudio", "r script"]},
    "sql":              {"category": "Data Engineering", "difficulty": 2, "aliases": ["structured query language", "t-sql", "pl/sql", "plsql"]},
    "java":             {"category": "Programming",      "difficulty": 3, "aliases": ["core java", "j2ee", "java 8", "java 11"]},
    "scala":            {"category": "Programming",      "difficulty": 3, "aliases": []},
    "javascript":       {"category": "Programming",      "difficulty": 2, "aliases": ["js", "node.js", "nodejs", "ecmascript", "es6"]},
    "typescript":       {"category": "Programming",      "difficulty": 2, "aliases": ["ts"]},
    "c++":              {"category": "Programming",      "difficulty": 3, "aliases": ["cpp", "c/c++"]},
    "go":               {"category": "Programming",      "difficulty": 3, "aliases": ["golang", "go-lang"]},
    "rust":             {"category": "Programming",      "difficulty": 4, "aliases": ["rustlang"]},
    "bash":             {"category": "Programming",      "difficulty": 2, "aliases": ["bash scripting", "bourne again shell"]},
    "shell":            {"category": "Programming",      "difficulty": 2, "aliases": ["shell scripting", "sh", "zsh"]},

    # ── ML / AI ────────────────────────────────
    "machine learning": {"category": "ML/AI",            "difficulty": 3, "aliases": ["ml", "machine-learning"]},
    "deep learning":    {"category": "ML/AI",            "difficulty": 4, "aliases": ["dl", "deep-learning"]},
    "nlp":              {"category": "ML/AI",            "difficulty": 4, "aliases": ["natural language processing", "text processing", "nlu", "nlg"]},
    "computer vision":  {"category": "ML/AI",            "difficulty": 4, "aliases": ["cv", "machine vision"]},
    "reinforcement learning": {"category": "ML/AI",      "difficulty": 4, "aliases": ["rl", "deep reinforcement learning"]},
    "llm":              {"category": "ML/AI",            "difficulty": 4, "aliases": ["large language models", "large language model", "llms", "generative ai", "genai", "foundation models"]},
    "pytorch":          {"category": "ML/AI",            "difficulty": 3, "aliases": ["py-torch", "torch"]},
    "tensorflow":       {"category": "ML/AI",            "difficulty": 3, "aliases": ["tf", "tensor-flow", "tensorflow 2"]},
    "keras":            {"category": "ML/AI",            "difficulty": 2, "aliases": ["tf.keras"]},
    "scikit-learn":     {"category": "ML/AI",            "difficulty": 2, "aliases": ["sklearn", "scikit learn", "sci-kit learn"]},
    "xgboost":          {"category": "ML/AI",            "difficulty": 2, "aliases": ["xg-boost", "extreme gradient boosting"]},
    "hugging face":     {"category": "ML/AI",            "difficulty": 3, "aliases": ["huggingface", "hf"]},
    "transformers":     {"category": "ML/AI",            "difficulty": 4, "aliases": ["transformer models", "attention mechanisms"]},

    # ── Data Engineering ───────────────────────
    "spark":            {"category": "Data Engineering", "difficulty": 3, "aliases": ["apache spark", "pyspark", "spark sql"]},
    "hadoop":           {"category": "Data Engineering", "difficulty": 3, "aliases": ["apache hadoop", "hdfs", "mapreduce"]},
    "kafka":            {"category": "Data Engineering", "difficulty": 3, "aliases": ["apache kafka"]},
    "airflow":          {"category": "Data Engineering", "difficulty": 3, "aliases": ["apache airflow"]},
    "dbt":              {"category": "Data Engineering", "difficulty": 2, "aliases": ["data build tool"]},
    "etl":              {"category": "Data Engineering", "difficulty": 2, "aliases": ["extract, transform, load", "extract transform load", "elt"]},
    "data pipeline":    {"category": "Data Engineering", "difficulty": 3, "aliases": ["data pipelines", "data pipelining"]},
    "data warehouse":   {"category": "Data Engineering", "difficulty": 3, "aliases": ["data warehousing", "dwh", "edw"]},
    "snowflake":        {"category": "Data Engineering", "difficulty": 2, "aliases": ["snowflake db", "snow flake"]},
    "redshift":         {"category": "Data Engineering", "difficulty": 2, "aliases": ["amazon redshift", "aws redshift"]},
    "bigquery":         {"category": "Data Engineering", "difficulty": 2, "aliases": ["google bigquery", "gcp bigquery", "bq"]},
    "databricks":       {"category": "Data Engineering", "difficulty": 3, "aliases": ["data bricks"]},

    # ── Cloud ──────────────────────────────────
    "aws":              {"category": "Cloud",            "difficulty": 2, "aliases": ["amazon web services", "amazon cloud"]},
    "azure":            {"category": "Cloud",            "difficulty": 2, "aliases": ["microsoft azure", "ms azure"]},
    "gcp":              {"category": "Cloud",            "difficulty": 2, "aliases": ["google cloud platform", "google cloud"]},
    "docker":           {"category": "Cloud/DevOps",     "difficulty": 2, "aliases": ["docker containers", "dockerization"]},
    "kubernetes":       {"category": "Cloud/DevOps",     "difficulty": 3, "aliases": ["k8s", "kube", "k3s"]},
    "terraform":        {"category": "Cloud/DevOps",     "difficulty": 3, "aliases": ["hashicorp terraform"]},
    "ci/cd":            {"category": "Cloud/DevOps",     "difficulty": 2, "aliases": ["continuous integration", "continuous deployment", "continuous delivery", "cicd", "ci cd"]},

    # ── Statistics / Analytics ─────────────────
    "statistics":       {"category": "Analytics",        "difficulty": 2, "aliases": ["stats", "statistical analysis", "statistical modeling"]},
    "a/b testing":      {"category": "Analytics",        "difficulty": 2, "aliases": ["ab testing", "a/b test", "split testing"]},
    "hypothesis testing": {"category": "Analytics",      "difficulty": 2, "aliases": ["statistical hypothesis testing"]},
    "regression":       {"category": "Analytics",        "difficulty": 2, "aliases": ["linear regression", "logistic regression"]},
    "time series":      {"category": "Analytics",        "difficulty": 3, "aliases": ["time series analysis", "time-series", "forecasting"]},
    "tableau":          {"category": "Analytics",        "difficulty": 1, "aliases": ["tableau desktop", "tableau server"]},
    "power bi":         {"category": "Analytics",        "difficulty": 1, "aliases": ["powerbi", "ms power bi", "microsoft power bi"]},
    "excel":            {"category": "Analytics",        "difficulty": 1, "aliases": ["microsoft excel", "ms excel", "excel vba"]},

    # ── Databases ──────────────────────────────
    "postgresql":       {"category": "Database",         "difficulty": 2, "aliases": ["postgres", "postgre-sql"]},
    "mysql":            {"category": "Database",         "difficulty": 2, "aliases": ["my-sql"]},
    "mongodb":          {"category": "Database",         "difficulty": 2, "aliases": ["mongo", "mongo db"]},
    "redis":            {"category": "Database",         "difficulty": 2, "aliases": ["redis cache"]},
    "elasticsearch":    {"category": "Database",         "difficulty": 3, "aliases": ["elastic search", "elk"]},
    "neo4j":            {"category": "Database",         "difficulty": 3, "aliases": ["graph database", "neo 4j"]},

    # ── Soft / Process ─────────────────────────
    "agile":            {"category": "Process",          "difficulty": 1, "aliases": ["agile methodology", "agile methodologies"]},
    "scrum":            {"category": "Process",          "difficulty": 1, "aliases": ["scrum master", "scrum framework"]},
    "git":              {"category": "Tools",            "difficulty": 1, "aliases": ["github", "gitlab", "bitbucket", "version control"]},
    "linux":            {"category": "Tools",            "difficulty": 2, "aliases": ["unix", "ubuntu", "centos", "debian"]},
    "communication":    {"category": "Soft Skills",      "difficulty": 1, "aliases": ["written communication", "verbal communication", "presentation skills"]},
    "leadership":       {"category": "Soft Skills",      "difficulty": 2, "aliases": ["team leadership", "mentoring", "management"]},
    "problem solving":  {"category": "Soft Skills",      "difficulty": 2, "aliases": ["troubleshooting", "analytical skills"]},
}

# Signals that boost relevance score of the NEXT noun/skill phrase
IMPORTANCE_SIGNALS: list[str] = [
    "required", "must have", "essential", "key skill", "strong",
    "proficient", "expert", "hands-on", "proven", "extensive",
    "minimum", "at least", "years of", "experience with",
    "knowledge of", "familiarity with", "background in",
]

In [4]:
class JDSkillExtractor:
    def __init__(self, nlp_base, expanded_vocab):
        self.nlp_base = nlp_base
        self.expanded_vocab = expanded_vocab
        self.taxonomy = SKILL_TAXONOMY
        
        self.alias_to_canonical = {}
        
        # OPTIMIZATION 2.1: Pre-compile regular expressions for Engine 1 (Taxonomy).
        # Instead of compiling regex thousands of times per row, we compile them ONCE during initialization.
        self.taxonomy_compiled_regex = {}
        for canonical, info in self.taxonomy.items():
            canonical_lower = canonical.lower()
            self.alias_to_canonical[canonical_lower] = canonical_lower
            
            search_terms = [canonical] + info.get("aliases", [])
            compiled_patterns = []
            for term in search_terms:
                term_lower = term.lower()
                self.alias_to_canonical[term_lower] = canonical_lower
                
                # Pre-compile the exact word boundary match pattern
                pattern = r"(?<![a-zA-Z])" + re.escape(term_lower) + r"(?![a-zA-Z])"
                compiled_patterns.append(re.compile(pattern))
                
            self.taxonomy_compiled_regex[canonical_lower] = compiled_patterns

        # OPTIMIZATION 2.2: Pre-compile regular expressions for Engine 2 (Word2Vec).
        # This eliminates ~2.6 million redundant regex compilations for a 500-row dataset.
        self.expanded_vocab_compiled_regex = {}
        for skill in self.expanded_vocab:
            skill_key = skill.lower()
            pattern = r"(?<![a-zA-Z])" + re.escape(skill_key) + r"(?![a-zA-Z])"
            self.expanded_vocab_compiled_regex[skill_key] = re.compile(pattern)
        
        self.importance_weights = {
            "required": 0.8, "must have": 0.8, "essential": 0.8, "key skill": 0.8, 
            "strong": 0.6, "expert": 0.6, "proficient": 0.5, "hands-on": 0.4
        }
        self.difficulty_map = {"Basic": 1, "Intermediate": 2, "Senior": 3, "Expert": 4}
        self.context_signals = {"expert": "Expert", "advanced": "Expert", "proficient": "Intermediate", "basic": "Basic"}

    # OPTIMIZATION 3: Accept the pre-parsed spacy `doc` object instead of a raw text string.
    # This prevents running the heavy spaCy pipeline multiple times for the same Job Description.
    def _get_dynamic_difficulty_and_relevance(self, doc, canonical_skill: str, base_diff: int):
        search_terms = [canonical_skill.lower()] + self.taxonomy.get(canonical_skill.lower(), {}).get("aliases", [])
        
        dyn_level = None
        relevance_multiplier = 1.0 
        
        # Iterate through the already-parsed sentences in the doc object
        for sent in doc.sents:
            sent_text_lower = sent.text.lower()
            if any(term in sent_text_lower for term in search_terms):
                for token in sent:
                    if token.text.lower() in self.context_signals:
                        dyn_level = self.context_signals[token.text.lower()]
                for signal, weight in self.importance_weights.items():
                    if signal in sent_text_lower:
                        relevance_multiplier += weight
                        
        relevance_multiplier = min(round(relevance_multiplier, 2), 3.0)
        if not dyn_level:
            reverse_diff_map = {1: "Basic", 2: "Intermediate", 3: "Senior", 4: "Expert"}
            dyn_level = reverse_diff_map.get(base_diff, "Intermediate")
            
        return dyn_level, relevance_multiplier

    def extract_skills(self, jd_text: str) -> Dict[str, dict]:
        jd_lower = jd_text.lower()
        extracted = {}

        # OPTIMIZATION 4: Parse the entire JD text through spaCy ONLY ONCE per record.
        doc = self.nlp_base(jd_lower)

        # ENGINE 1: Canonical Dictionary & Alias Extraction (Using Pre-compiled Regex)
        for canonical_skill, compiled_patterns in self.taxonomy_compiled_regex.items():
            for compiled_pattern in compiled_patterns:
                # Direct regex search using the pre-compiled pattern (Lightning fast)
                if compiled_pattern.search(jd_lower):
                    extracted[canonical_skill] = {
                        "name": canonical_skill.title(), 
                        "base_diff": self.taxonomy[canonical_skill]["difficulty"]
                    }
                    break 

        # ENGINE 2: Word2Vec Expanded Semi-Supervised Vocabulary (Using Pre-compiled Regex)
        for skill_key, compiled_pattern in self.expanded_vocab_compiled_regex.items():
            # Avoid overwriting skills already found by Engine 1
            if skill_key not in extracted and skill_key not in self.alias_to_canonical:
                if compiled_pattern.search(jd_lower):
                    # Assign a default intermediate difficulty (2) for auto-discovered skills
                    extracted[skill_key] = {"name": skill_key.title(), "base_diff": 2}

        # MERGE & CALCULATE ROADMAP SCORES
        final_jd_skills = {}
        for canonical_key, data in extracted.items():
            # Pass the pre-parsed `doc` object to the helper function
            dyn_level, relevance_multiplier = self._get_dynamic_difficulty_and_relevance(doc, canonical_key, data["base_diff"])
            difficulty_val = self.difficulty_map.get(dyn_level, data["base_diff"])
            
            # ROADMAP MATH: Ensure lower difficulty gets higher priority score
            learning_weight = 5 - difficulty_val
            p_score = round(relevance_multiplier * learning_weight, 2)
            
            final_jd_skills[canonical_key] = {
                "name": data["name"], 
                "target_level": dyn_level, 
                "priority_score": p_score
            }

        return final_jd_skills

# Instantiate Extractor passing the expanded_vocab instead of nlp_custom_ner
jd_extractor = JDSkillExtractor(nlp_base, expanded_vocab)
print("✅ Phase 3 Optimized Complete: Dual-Engine Extractor initialized.")

✅ Phase 3 Optimized Complete: Dual-Engine Extractor initialized.


## Phase 3: Define the Gap Analysis Pipeline (with Alias Mapping)
When comparing the candidate's skills against the JD, we must check if the candidate has the canonical skill *or* any of its accepted aliases before marking it as a "Missing Skill".

In [5]:
def clean_text(text: str) -> str:
    """Removes irregular characters and compresses spaces to prevent SkillNER from crashing."""
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'[^a-zA-Z0-9\s.,;-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_resume_skills(text: str) -> set:
    """Extracts skills from a resume with crash-protection."""
    cleaned_text = clean_text(text)
    if not cleaned_text:
        return set()
    
    extracted_skills = set()
    try:
        annotations = skill_extractor.annotate(cleaned_text)
        
        for match in annotations.get('results', {}).get('full_matches', []):
            extracted_skills.add(match['doc_node_value'].lower())
        for match in annotations.get('results', {}).get('ngram_scored', []):
            extracted_skills.add(match['doc_node_value'].lower())
            
    except Exception:
        # Silently pass if SkillNER encounters an internal n-gram error.
        # The fallback regex in the next step will catch remaining keywords.
        pass
        
    return extracted_skills

def calculate_skill_gap(row) -> dict:
    """Calculates the missing skills between JD and Resume, returning prioritization details."""
    jd_text = str(row.get('job_description_text', ''))
    resume_text = str(row.get('resume_text', ''))
    
    if not jd_text.strip():
        return {}
        
    # Extract skills from JD and Resume
    jd_skills_dict = jd_extractor.extract_skills(jd_text)
    resume_skills_set = extract_resume_skills(resume_text)
    
    skill_extracted = {}
    resume_text_lower = resume_text.lower()
    
    for canonical_key, info in jd_skills_dict.items():
        search_terms = [canonical_key] + jd_extractor.taxonomy.get(canonical_key, {}).get("aliases", [])
        
        # 1. Match against SkillNER output
        is_matched = any(term in resume_skills_set for term in search_terms)
        
        # 2. Fallback Regex Match (Using pre-compiled patterns from the jd_extractor if available)
        if not is_matched:
            # Check if this skill exists in our pre-compiled taxonomy regex map to save compilation time
            compiled_patterns = jd_extractor.taxonomy_compiled_regex.get(canonical_key, [])
            if compiled_patterns:
                for compiled_pattern in compiled_patterns:
                    if compiled_pattern.search(resume_text_lower):
                        is_matched = True
                        break
            else:
                # Standard runtime regex for Word2Vec expanded skills
                pattern = r"(?<![a-zA-Z])" + re.escape(canonical_key) + r"(?![a-zA-Z])"
                if re.search(pattern, resume_text_lower):
                    is_matched = True
                
        # 3. Compile gap analysis dict for missing skills
        if not is_matched:
            skill_extracted[info["name"]] = {
                "priority_score": info["priority_score"],
                "difficulty_level": info["target_level"]
            }
            
    return skill_extracted

print("✅ Phase 4 Optimized Complete: Pipeline functions defined.")

✅ Phase 4 Optimized Complete: Pipeline functions defined.


## Phase 4: Execution & Export (Alias Testing)
We will test the pipeline using a mock dataset designed to trigger the alias logic. 
* **Scenario 1**: The JD asks for abbreviations (`AWS`, `LLMs`), but the candidate wrote the full names (`Amazon Web Services`, `Large Language Models`).
* **Scenario 2**: The JD asks for full names/different aliases (`JS`, `Machine Learning`), and the candidate wrote (`JavaScript`).

The gap analysis should successfully recognize these as matches and *not* penalize the candidate.

In [6]:
# 1. Load Data
print("Initializing dataset...")
# NOTE: Replace the path below with your actual CSV file loading method.
df = pd.read_csv("/Users/fengmengting/Desktop/berkeley/26spring/INDENG243 Analytics Lab/project/module2_new/final_skill_extraction/job_des_full.csv").head(500)

output_filename = "final_gap_analysis.jsonl"
chunk_size = 10

# --- NEW: Breakpoint Resume Logic ---
start_index = 0

# Check if the file already exists from a previous interrupted run
if os.path.exists(output_filename):
    # Count the number of lines (records) already processed
    with open(output_filename, 'r', encoding='utf-8') as f:
        start_index = sum(1 for line in f)
    print(f"⚠️ Found existing file. Skipping the first {start_index} records...")
else:
    print("No existing progress found. Starting from scratch...")

# Check if everything is already done
if start_index >= len(df):
    print("✅ All records have already been processed!")
else:
    print(f"Processing gap analysis for remaining {len(df) - start_index} records in chunks of {chunk_size}...")

    # 2. Iterate through the DataFrame starting from start_index
    for i in tqdm(range(start_index, len(df), chunk_size), desc="Processing Chunks"):
        # Extract the current chunk
        chunk = df.iloc[i:i + chunk_size].copy()
        
        # Apply your skill extraction function to this chunk 
        chunk['skill_extracted'] = chunk.apply(calculate_skill_gap, axis=1)
        
        # 3. Format and Export this specific chunk
        final_chunk = chunk[['job_description_text', 'resume_text', 'skill_extracted']]
        
        # Append directly to the JSONL file
        final_chunk.to_json(
            output_filename, 
            orient="records", 
            lines=True, 
            mode='a', 
            force_ascii=False
        )

    print(f"\n✅ Pipeline Execution Complete. Data successfully saved to {output_filename}")

Initializing dataset...
⚠️ Found existing file. Skipping the first 240 records...
Processing gap analysis for remaining 260 records in chunks of 10...


Processing Chunks: 100%|██████████| 26/26 [1:27:58<00:00, 203.02s/it]


✅ Pipeline Execution Complete. Data successfully saved to final_gap_analysis.jsonl
